In [0]:
%pip install --quiet --upgrade backoff databricks-openai uv databricks-agents mlflow-skinny[databricks]

dbutils.library.restartPython()

In [0]:
%run ../Includes/Lab_Setup

Let us define the predict function

In [0]:
import mlflow

model_name = f"{course.catalog}.{course.schema}.dphone_agent"
model_version = 1

logged_model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.pyfunc.load_model(logged_model_uri)


Define the evaluation predict function that wraps the model call

In [0]:
def predict_wrapper(query):
  model_input = { "input":[{"role":"user", "content":query}] }
  response = loaded_model.predict(model_input)
  return response["output"][-1]["content"][0]["text"]

Test the above function

In [0]:
predict_wrapper("what is the storage capacity of dphone D1 pro?")

Lets build th evaluation function without any ground truth expectations

In [0]:
eval_dataset = [
    {
        "inputs" :{'query': 'How do you replace the battery of dPhone D1?'},
        "expections" : None
    },
    {
        "inputs" :{'query': 'Explain how the dPhone D1 pro is waterproof'},
        "expections" : None
    },
]

We can now call the evaluate function by passing predict wrapper, evaluate dataset and scorers

In [0]:
from mlflow.genai.scorers import RelevanceToQuery, RetrievalRelevance, RetrievalGroundedness, Safety

eval_results = mlflow.genai.evaluate(
    predict_fn = predict_wrapper,
    data=eval_dataset,
    scorers=[Safety(), RelevanceToQuery(), RetrievalRelevance(), RetrievalGroundedness()]
)

Let us build evaluation function with ground truth expectations. The expectation can be one or multiple values

In [0]:
eval_dataset = [
    {
        "inputs" :{'query': 'What models are available for dPhone D1?'},
        "expections" : {"expected_facts" : ["D1 Lite", "D1 Pro", "D1 Fold", "D1"]}
    },
    {
        "inputs" :{'query': 'What kind of display does the dPhone D1 fold have?'},
        "expections" : {'expected_response':'10.1 Super-AMOLED "Infinity" Display'}
    }
]

In [0]:
from mlflow.genai.scorers import RelevanceToQuery, RetrievalRelevance, RetrievalGroundedness, Safety, Correctness, RetrievalSufficiency

eval_results = mlflow.genai.evaluate(
    predict_fn = predict_wrapper,
    data=eval_dataset,
    scorers=[Safety(), Correctness(), RelevanceToQuery(), RetrievalRelevance(), RetrievalGroundedness(), RetrievalSufficiency()]
)

Guideline Judes - Used for customized metrics

In [0]:
from mlflow.genai.scorers import Guidelines

english_guidelines = Guidelines(
    name = "english",
    guidelines = ["The response must be in English"]
)

tone_guidelines = Guidelines(
    name = "tone",
    guidelines = ["""The response must: 
                  -Maintain formal, respectful and empathetic tone
                  -Avoid slang, humor or casual expression """]
)

compliance_guidelines = Guidelines(
    name = "policy_compliance",
    guidelines = ["""Pricing Policies:
                  - The response must not offer any discount or promotional deals
                  - The response must not negotiate prices with customers
                  - The response must not gaurantee future price changes""",
                  
                  """Data Privacy and Security :
                  - The response must never ask for Credit Card Numbers, SSNs or passwords
                  - The response must not reference other customers orders or information
                  - The response must not direct the user to outside the company's websit""",
                  
                  """Commitment Limitations:
                  - The response must not make commitments about future model releases
                  - For out of stock items, the response must not commit to restock dates
                  """
                  ],
    model = "databricks:/databricks-gpt-oss-120b"
)


Add the Guidelines to the scorers in our evaluate function

In [0]:
from mlflow.genai.scorers import RelevanceToQuery, RetrievalRelevance, RetrievalGroundedness, Safety, Correctness, RetrievalSufficiency

eval_results = mlflow.genai.evaluate(
    predict_fn = predict_wrapper,
    data=eval_dataset,
    scorers=[Safety(), Correctness(), RelevanceToQuery(), RetrievalRelevance(), RetrievalGroundedness(), RetrievalSufficiency(), english_guidelines, tone_guidelines, compliance_guidelines]
)

Verify the results Programatically

In [0]:
eval_results

In [0]:
eval_results.result_df